# Phase 0.6 follow-ups — F2 (camera embedding) and F3 (idle-trimmed data)

Companion to `Details/phase06_camera_ablation_training.md` §8. Everything is pulled from the Hub.

| | What it tests | Runs |
|---|---|---|
| **F2** | Is `top+wrist`'s reach deficit an *arbitration* problem? ACT gives both cameras identical positional embeddings, so the model can only tell the streams apart by appearance. Add a learned per-camera identity vector and retrain. | 1 × 60k (`both`) |
| **F3** | Does idle-trimming let short action chunks work? The untrimmed set forced `n_action_steps=100`, i.e. ~9 observations per 30 s episode, which blunts any camera ablation. | 3 × 60k (`top`, `wrist`, `both`) |

Recipe identical to the 60k baselines — same 45/5 balanced holdout, seed 1000, `batch_size=8`,
`eval_steps=5000`, fp32, no augmentation — into the same wandb project, so the curves overlay
`act_top_s1000` / `act_wrist_s1000` / `act_both_s1000`.

## How to use this notebook

**Part A — Setup.** Run cells 1–4 once.
**Part B — Smoke test.** One cell. Tests **both** datasets end to end, prints a readiness verdict and the
projected total runtime. Paste that output before starting the long run.
**Part C — Training.** One cell. Runs F2 and F3 to completion and uploads everything. Nothing else to click.
**Part D — Recovery.** Individual cells, only if something fails.

Part C refuses to start unless Part B passed, so a bad token or a missing dataset fails in fifteen
minutes rather than at hour four.

# Part A — Setup

## 1 — Runtime check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", GPU)
assert GPU, "No GPU. Runtime > Change runtime type > GPU (L4)."
if "T4" in GPU:
    print("\n!! T4: this workload is ~3x slower here and the programme would exceed 30 h. Prefer L4.")
elif "A100" in GPU:
    print("\nNote: A100 costs ~2.5x the compute units for ~1.7x the speed. L4 is the better trade;")
    print("      VRAM is irrelevant here (peak 3.5 GiB of 24).")

## 2 — Install

From the **v0.6.1 source tree**, not the wheel, because F2 edits `modeling_act.py`. A few minutes.

In [ ]:
!git clone --depth 1 --branch v0.6.1 https://github.com/huggingface/lerobot.git /content/lerobot
%pip install -q -e /content/lerobot
%pip install -q wandb
import lerobot; print("lerobot", lerobot.__version__)

> If `import lerobot` fails immediately after the install, restart the runtime and re-run this cell.

## 3 — Authentication

**A Colab secret alone is not enough.** `huggingface_hub` resolves a token as OIDC → `HF_TOKEN` env var →
token file → **Colab secret**, and that last path calls `google.colab.userdata.get()`, which needs the
kernel's channel to the Colab frontend. `lerobot-train` is a **subprocess**, where that channel does not
exist — so with only a secret set, `whoami()` succeeds in the notebook while training authenticates as
nobody (401 on a private dataset, 401 on `create_repo`).

These cells copy the token into the environment *and* the token file, then prove a subprocess can use it.

In [ ]:
import os, sys, subprocess
from huggingface_hub import get_token, login, whoami

tok = os.environ.get("HF_TOKEN") or get_token()
if not tok:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        tok = None
if not tok:
    from huggingface_hub import notebook_login
    notebook_login(); tok = get_token()
assert tok, "no HF token — add a Colab secret named HF_TOKEN, or run notebook_login()"

os.environ["HF_TOKEN"] = tok                      # inherited by every subprocess
login(token=tok, add_to_git_credential=False)     # also writes ~/.cache/huggingface/token

r = subprocess.run([sys.executable, "-c",
                    "from huggingface_hub import whoami; print(whoami()['name'])"],
                   capture_output=True, text=True)
assert r.returncode == 0, "a subprocess still cannot authenticate:\n" + r.stderr[-800:]
print("HF ok — notebook:", whoami()["name"], "| subprocess:", r.stdout.strip())

a = whoami().get("auth", {}).get("accessToken", {})
if a.get("role") == "fineGrained":
    g = (a.get("fineGrained") or {}).get("global", [])
    print("token: fine-grained | global scopes:", g)
    assert any("repo.write" in s for s in g), (
        "This token has no GLOBAL repo-write scope, so it cannot CREATE the model repos these runs push "
        "to — you would get a 401 at the first checkpoint. Fix: huggingface.co/settings/tokens -> edit "
        "-> tick 'Write access to contents/settings of all repos', or use a classic Write token.")
else:
    print("token role:", a.get("role"), "('write' can create repos, 'read' cannot)")

In [ ]:
import os, sys, subprocess, wandb

key = os.environ.get("WANDB_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        key = None
if key:
    os.environ["WANDB_API_KEY"] = key; wandb.login(key=key)
else:
    wandb.login()                                  # key from https://wandb.ai/authorize
    os.environ["WANDB_API_KEY"] = wandb.api.api_key

r = subprocess.run([sys.executable, "-c", "import wandb; print(bool(wandb.api.api_key))"],
                   capture_output=True, text=True)
assert r.stdout.strip() == "True", "a subprocess cannot see the wandb key:\n" + r.stderr[-500:]
print("wandb ok in notebook and subprocess")

## 4 — Configuration

The holdout must match the baselines exactly. The 50 episodes were recorded one position at a time
(`ep 0-4 = P1 … 45-49 = P10`), so lerobot's default "last N episodes" split would hold out **all of P10**
and remove that position from training. Reordering `--dataset.episodes` puts a balanced holdout in the
tail instead: the last episode of P2, P4, P6, P8, P10.

In [ ]:
import collections, re, subprocess, sys, time

HF_USER = "HALDijkstraaa"
DATASET = f"{HF_USER}/so101_toolkit_cylinder_20260917_165544"
TRIMMED = f"{DATASET}_trimmed"
PROJECT = "phase06-camera-ablation"
SEED, STEPS, BATCH = 1000, 60_000, 8
BASE_PARAMS = 51_597_190          # stock ACT, 2 cameras; +1024 with the F2 patch

HOLDOUT  = [9, 19, 29, 39, 49]
EPISODES = [e for e in range(50) if e not in HOLDOUT] + HOLDOUT

STATE = "'observation.state': {'type': 'STATE', 'shape': [6]}"
CAM   = lambda c: f"'observation.images.{c}': {{'type': 'VISUAL', 'shape': [3, 480, 640]}}"
FEATS = {"top":   f"{{{STATE}, {CAM('top')}}}",
         "wrist": f"{{{STATE}, {CAM('wrist')}}}",
         "both":  f"{{{STATE}, {CAM('top')}, {CAM('wrist')}}}"}

def train_cmd(cond, job, dataset=DATASET, steps=STEPS, extra=()):
    """Checkpoints go to the Hub at every save_freq, so a disconnect costs at most 10k steps."""
    return ["lerobot-train",
        f"--dataset.repo_id={dataset}", f"--dataset.episodes={EPISODES}",
        "--dataset.eval_split=0.1", "--eval_steps=5000",
        "--policy.type=act", "--policy.device=cuda",
        f"--policy.input_features={FEATS[cond]}",
        "--policy.push_to_hub=true", f"--policy.repo_id={HF_USER}/{job}", "--policy.private=true",
        "--save_checkpoint_to_hub=true",
        f"--batch_size={BATCH}", f"--steps={steps}", "--num_workers=8", f"--seed={SEED}",
        "--save_freq=10000", "--log_freq=200",
        f"--output_dir=/content/outputs/{job}", f"--job_name={job}",
        "--wandb.enable=true", f"--wandb.project={PROJECT}", *extra]

KEEP = ("step:", "eval_loss", "Train/eval split", "num_learnable_params",
        "Traceback", "Error", "error:", "Track this run", "End of training")

def run(cmd, expect_params=None, quiet=True):
    """Stream a run, dropping tqdm spam so an 11 h session does not drown the browser.

    expect_params guards against training the wrong model variant (see use_camera_embed)."""
    t0, seen, tail = time.time(), None, collections.deque(maxlen=60)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        tail.append(line)
        m = re.search(r"num_learnable_params=(\d+)", line)
        if m: seen = int(m.group(1))
        if (not quiet) or any(k in line for k in KEEP) or "wandb.ai" in line:
            sys.stdout.write(line); sys.stdout.flush()
    p.wait()
    dt = (time.time() - t0) / 3600
    print(f"[exit {p.returncode} after {dt:.2f} h]")
    if p.returncode != 0:
        print("--- last lines ---"); print("".join(tail)[-2000:])
    if expect_params is not None and seen is not None and seen != expect_params:
        raise RuntimeError(f"wrong model variant: trained {seen:,} params, expected {expect_params:,}")
    return {"code": p.returncode, "hours": dt, "params": seen}

print("holdout:", HOLDOUT, "| episode-list tail:", EPISODES[-6:])

---
# Part B — Smoke test

One cell, ~15 minutes. Runs 300 real training steps against **each** dataset — the balanced split,
periodic validation, wandb, and (once) a checkpoint pushed to the Hub — then checks every result and
prints a readiness verdict plus the projected runtime for the full programme.

It fails loudly rather than warning, so Part C cannot start on a broken setup.

In [ ]:
from huggingface_hub import HfApi

SMOKE_REPO, N = f"{HF_USER}/act_smoke_delete_me", 300
api, results, t_smoke = HfApi(), {}, time.time()

def smoke(dataset, push):
    job = "smoke"
    subprocess.run(["rm", "-rf", f"/content/outputs/{job}"], check=False)
    cmd = [c for c in train_cmd("both", job, dataset=dataset, steps=N)
           if not c.startswith(("--policy.repo_id", "--save_freq", "--eval_steps"))]
    cmd += [f"--policy.repo_id={SMOKE_REPO}", f"--eval_steps={N // 2}", f"--save_freq={N}"]
    if not push:
        # a checkpoint is always written at the final step, so the Hub push must be disabled outright
        cmd = [c for c in cmd if not c.startswith(("--policy.push_to_hub", "--save_checkpoint_to_hub"))]
        cmd += ["--policy.push_to_hub=false", "--save_checkpoint_to_hub=false"]
    out = subprocess.run(cmd, capture_output=True, text=True)
    text = out.stdout + out.stderr
    rates = [float(x) for x in re.findall(r"([\d.]+)step/s", text)]
    checks = [("training completed", out.returncode == 0),
              ("balanced split 45 train / 5 eval", "45 train, 5 eval" in text),
              ("both cameras in input_features",
               "observation.images.top" in text and "observation.images.wrist" in text),
              ("validation ran", "eval_loss=" in text),
              ("wandb connected", "wandb.ai" in text)]
    m = re.search(r"num_learnable_params=(\d+)", text)
    checks.append((f"stock ACT params ({m.group(1) if m else '?'})",
                   m is not None and int(m.group(1)) == BASE_PARAMS))
    if push:
        try:
            files = api.list_repo_files(SMOKE_REPO, repo_type="model")
            checks.append(("checkpoint uploaded to the Hub",
                           any(f"checkpoints/{N:06d}/" in f for f in files)))
        except Exception as e:
            checks.append((f"checkpoint uploaded to the Hub ({type(e).__name__})", False))
    subprocess.run(["rm", "-rf", f"/content/outputs/{job}"], check=False)
    return {"checks": checks, "rate": sorted(rates)[len(rates) // 2] if rates else None,
            "tail": text[-2500:]}

print("checking both datasets exist and are readable...")
missing = [d for d in (DATASET, TRIMMED) if not api.repo_exists(d, repo_type="dataset")]
if missing:
    raise RuntimeError(f"dataset(s) not found or not readable with this token: {missing}\n"
                       "Push the trimmed set first:  ./scripts/push_dataset.sh ~/trimmed <repo_id>")

for label, dataset, push in (("original", DATASET, True), ("trimmed", TRIMMED, False)):
    print(f"\n{'=' * 64}\n  smoke: {label}  ({dataset}){'  [+ Hub push]' if push else ''}\n{'=' * 64}")
    results[label] = smoke(dataset, push)
    for name, ok in results[label]["checks"]:
        print(f"  {'PASS' if ok else 'FAIL'}  {name}")
    if not all(ok for _, ok in results[label]["checks"]):
        print("\n--- output tail ---\n" + results[label]["tail"])

rates = [r["rate"] for r in results.values() if r["rate"]]
ok_all = all(ok for r in results.values() for _, ok in r["checks"]) and rates
print(f"\n{'=' * 64}")
if rates:
    r2 = sum(rates) / len(rates)                 # two-camera it/s
    r1 = r2 * 1.95                                # one camera ~2x (locally 134 vs 72 ms/step)
    ev  = 12 * (0.1 * 34000 / BATCH) / (3 * r2) / 3600     # 12 validation passes, forward-only
    f2  = STEPS / r2 / 3600 + ev
    f3  = 2 * (STEPS / r1 / 3600 + ev / 2) + STEPS / r2 / 3600 + ev
    print(f"  GPU            : {GPU}")
    print(f"  measured       : {r2:.2f} it/s two-camera  ({r2 * BATCH:.0f} samples/s)")
    print(f"  F2  both       : {f2:4.1f} h")
    print(f"  F3  top        : {STEPS / r1 / 3600 + ev / 2:4.1f} h")
    print(f"  F3  wrist      : {STEPS / r1 / 3600 + ev / 2:4.1f} h")
    print(f"  F3  both       : {STEPS / r2 / 3600 + ev:4.1f} h")
    print(f"  TOTAL Part C   : {f2 + f3:4.1f} h")
    print(f"  smoke took     : {(time.time() - t_smoke) / 60:.0f} min")
SMOKE_OK = bool(ok_all)
print(f"\n  READY: {SMOKE_OK}")
print("=" * 64)
if SMOKE_OK:
    print(f"\nPaste the block above, then run Part C.")
    print(f"Clean up when convenient:  HfApi().delete_repo('{SMOKE_REPO}', repo_type='model')")
else:
    raise RuntimeError("smoke test failed — see the FAIL lines above. Do not start Part C.")

---
# Part C — Run everything

One click. Reverts/applies the ACT patch as needed, trains all four models, and uploads each one.

**Why the patch has to be toggled.** F2 edits `modeling_act.py` *on disk*, so it applies to every later
`lerobot-train` subprocess. Without an explicit revert, the three F3 runs would silently include the
camera embedding and idle-trimming would be confounded with F2. Every run also asserts its parameter
count, so the wrong variant fails immediately instead of producing a plausible-looking wrong answer.

A failed run is recorded and the next one starts anyway — the four runs are independent, and a transient
Hub error should not cost the night.

In [ ]:
import shutil, textwrap
from pathlib import Path
import lerobot.policies.act.modeling_act as _m

MODELING = Path(_m.__file__)
PRISTINE = MODELING.with_suffix(".py.pristine")
if not PRISTINE.exists():
    shutil.copy(MODELING, PRISTINE)

A_OLD = ("        if self.config.image_features:\n"
         "            self.encoder_img_feat_input_proj = nn.Conv2d(\n"
         "                backbone_model.fc.in_features, config.dim_model, kernel_size=1\n"
         "            )\n")
A_NEW = A_OLD + (
    "            # F2: learned identity vector per camera, zero-init so step 0 == upstream ACT.\n"
    "            self.camera_id_embed = nn.Embedding(len(self.config.image_features), config.dim_model)\n"
    "            nn.init.zeros_(self.camera_id_embed.weight)\n")
B_OLD = ("            for img in batch[OBS_IMAGES]:\n"
         '                cam_features = self.backbone(img)["feature_map"]\n'
         "                cam_pos_embed = self.encoder_cam_feat_pos_embed(cam_features).to(dtype=cam_features.dtype)\n"
         "                cam_features = self.encoder_img_feat_input_proj(cam_features)\n")
B_NEW = ("            for cam_idx, img in enumerate(batch[OBS_IMAGES]):\n"
         '                cam_features = self.backbone(img)["feature_map"]\n'
         "                cam_pos_embed = self.encoder_cam_feat_pos_embed(cam_features).to(dtype=cam_features.dtype)\n"
         "                cam_features = self.encoder_img_feat_input_proj(cam_features)\n"
         "                # F2: tag the block with which camera it came from (broadcasts over h, w)\n"
         "                cam_features = cam_features + self.camera_id_embed.weight[cam_idx].view(1, -1, 1, 1)\n")

_CHECK = textwrap.dedent("""
    from lerobot.policies.act.configuration_act import ACTConfig
    from lerobot.policies.act.modeling_act import ACTPolicy
    from lerobot.configs.types import FeatureType, PolicyFeature as PF
    cfg = ACTConfig(device="cpu")
    cfg.input_features = {"observation.state": PF(type=FeatureType.STATE, shape=(6,)),
                          "observation.images.top": PF(type=FeatureType.VISUAL, shape=(3, 480, 640)),
                          "observation.images.wrist": PF(type=FeatureType.VISUAL, shape=(3, 480, 640))}
    cfg.output_features = {"action": PF(type=FeatureType.ACTION, shape=(6,))}
    p = ACTPolicy(cfg)
    print(sum(x.numel() for x in p.parameters()),
          bool(getattr(p.model, "camera_id_embed", None) is not None
               and (p.model.camera_id_embed.weight == 0).all()))
""")

def use_camera_embed(enabled):
    """Switch the installed ACT between stock and F2-patched; verified in a subprocess."""
    shutil.copy(PRISTINE, MODELING)
    if enabled:
        s = PRISTINE.read_text()
        for old, new, where in ((A_OLD, A_NEW, "__init__"), (B_OLD, B_NEW, "forward")):
            assert s.count(old) == 1, f"anchor not found exactly once in {where} - version mismatch"
            s = s.replace(old, new)
        MODELING.write_text(s)
    r = subprocess.run([sys.executable, "-c", _CHECK], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-1500:]
    n, zero = r.stdout.split(); n = int(n)
    want = BASE_PARAMS + 1024 if enabled else BASE_PARAMS
    assert n == want, f"expected {want:,} params, got {n:,}"
    if enabled: assert zero == "True", "camera embedding must start at zero"
    print(f"  ACT is now {'F2-patched' if enabled else 'stock'}: {n:,} params")
    return n
print("use_camera_embed defined")

In [ ]:
assert globals().get("SMOKE_OK"), "run Part B first — it must print READY: True"

PLAN = [  # (label, camera-embed?, condition, dataset, job name)
    ("F2", True,  "both",  DATASET, f"act_both_s{SEED}_camemb"),
    ("F3", False, "top",   TRIMMED, f"act_top_s{SEED}_trim"),
    ("F3", False, "wrist", TRIMMED, f"act_wrist_s{SEED}_trim"),
    ("F3", False, "both",  TRIMMED, f"act_both_s{SEED}_trim"),
]

t_all, summary, state = time.time(), [], None
for label, patched, cond, dataset, job in PLAN:
    print(f"\n{'#' * 70}\n#  {label}  {cond:5s}  ->  {HF_USER}/{job}\n{'#' * 70}")
    t_run = time.time()
    try:
        if state != patched:
            use_camera_embed(patched); state = patched
        res = run(train_cmd(cond, job, dataset=dataset),
                  expect_params=BASE_PARAMS + (1024 if patched else 0))
        summary.append((label, job, "ok" if res["code"] == 0 else f"exit {res['code']}", res["hours"]))
    except Exception as e:
        print(f"!! {type(e).__name__}: {e}")
        summary.append((label, job, f"{type(e).__name__}", (time.time() - t_run) / 3600))

print(f"\n{'=' * 70}\n  SUMMARY after {(time.time() - t_all) / 3600:.1f} h\n{'=' * 70}")
for label, job, status, hours in summary:
    print(f"  {label}  {job:28s} {status:14s} {hours:5.2f} h   "
          f"https://huggingface.co/{HF_USER}/{job}")
print(f"\n  wandb project: {PROJECT}   (open it from wandb.ai/home)")
print("  Each run uploaded a checkpoint every 10k steps, tagged by step, plus a final model push.")
print("  Paste this summary back to review the results.")

---
# Part D — Recovery

Only if something in Part C failed.

**Resume a run from the Hub.** Checkpoints were pushed at every 10k steps, so at most 10k are lost. This
rejoins the same wandb run; step counter, optimizer state, episode order, holdout and seed all come back
from the checkpoint, and only `--steps` is read from the CLI:

```python
use_camera_embed(True)        # or False, matching the run you are resuming
run(["lerobot-train", f"--config_path={HF_USER}/act_both_s1000_camemb",
     "--resume=true", f"--steps={STEPS}", "--output_dir=/content/outputs/resumed"])
```

**Re-run one experiment.** Set the patch state, then the single run:

```python
use_camera_embed(False)
run(train_cmd("wrist", f"act_wrist_s{SEED}_trim", dataset=TRIMMED), expect_params=BASE_PARAMS)
```

**Recover a specific checkpoint** — each push is tagged with its step:
`--policy.pretrained_revision=030000`.

## What to compare afterwards

Everything lands in the **`phase06-camera-ablation`** wandb project next to the baselines.

| Compare | Against | Reading |
|---|---|---|
| `act_both_s1000_camemb` | `act_both_s1000` | A lower held-out loss is *encouraging but not the answer* — the baseline's deficit was in reach behavior, which action L1 barely sees. The robot decides |
| `act_*_s1000_trim` | each other | Expect a **higher** loss than the baselines: the trimmed set has no trivial "stay still" frames left to predict. That is a different validation set, not a regression — compare F3 runs to each other, not to the baselines |

Both experiments end at the robot. Pull the checkpoints with `hf download`, then run the §8 retest
protocol — interleaved against the relevant baseline in one session, scoring *reached the cylinder*
rather than success, and for F3 with a short `--policy.n_action_steps` (25), which is the whole point of
the trim.